In [1]:
import pandas as pd
import numpy as np 

FILE = "experiment_results.csv"

df = pd.read_csv(FILE)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print("\nColumns:")
print(df.columns.tolist())


DATASET OVERVIEW
Rows    : 14,000
Columns : 4

Columns:
['user_id', 'segment', 'variant', 'converted']


In [2]:
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(df.dtypes)


print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)

print(df.head())



DATA TYPES
user_id       int64
segment      object
variant      object
converted     int64
dtype: object

FIRST 5 ROWS
   user_id      segment    variant  converted
0   103792  paid_search    control          0
1   101683      organic  treatment          0
2   107268    app_store  treatment          0
3   108837  paid_search    control          0
4   101617  paid_search    control          0


In [3]:
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = df.isnull().sum()

print(missing)

if missing.sum() == 0:
    print("\nNo missing values found.")
else:
    print("\nMissing values exist — investigate before analysis.")



MISSING VALUES
user_id      0
segment      0
variant      0
converted    0
dtype: int64

No missing values found.


In [4]:
print("\n" + "=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)

duplicate_rows = df.duplicated().sum()
duplicate_users = df["user_id"].duplicated().sum()

print(f"Duplicate complete rows : {duplicate_rows}")
print(f"Duplicate user IDs      : {duplicate_users}")

print(f"Unique users            : {df['user_id'].nunique():,}")


DUPLICATE CHECK
Duplicate complete rows : 0
Duplicate user IDs      : 0
Unique users            : 14,000


In [5]:
expected_segments = {
    "organic",
    "paid_search",
    "referral",
    "app_store",
    "influencer"
}

expected_variants = {
    "control",
    "treatment"
}

expected_converted = {0, 1}


print("\n" + "=" * 60)
print("VALUE VALIDATION")
print("=" * 60)

actual_segments = set(df["segment"].unique())
actual_variants = set(df["variant"].unique())
actual_converted = set(df["converted"].unique())

print("Segments:")
print(sorted(actual_segments))

print("\nVariants:")
print(sorted(actual_variants))

print("\nConverted values:")
print(sorted(actual_converted))


invalid_segments = actual_segments - expected_segments
invalid_variants = actual_variants - expected_variants
invalid_converted = actual_converted - expected_converted

print("\nInvalid segments :", invalid_segments)
print("Invalid variants :", invalid_variants)
print("Invalid converted:", invalid_converted)





VALUE VALIDATION
Segments:
['app_store', 'influencer', 'organic', 'paid_search', 'referral']

Variants:
['control', 'treatment']

Converted values:
[np.int64(0), np.int64(1)]

Invalid segments : set()
Invalid variants : set()
Invalid converted: set()


In [6]:
print("\n" + "=" * 60)
print("SEGMENT DISTRIBUTION")
print("=" * 60)

segment_counts = df["segment"].value_counts()

segment_distribution = pd.DataFrame({
    "users": segment_counts,
    "share_pct": segment_counts / len(df) * 100
})

print(segment_distribution)



SEGMENT DISTRIBUTION
             users  share_pct
segment                      
paid_search   4812  34.371429
organic       4215  30.107143
referral      2838  20.271429
app_store     1885  13.464286
influencer     250   1.785714


In [7]:
print("\n" + "=" * 60)
print("OVERALL VARIANT DISTRIBUTION")
print("=" * 60)

variant_counts = df["variant"].value_counts()

variant_distribution = pd.DataFrame({
    "users": variant_counts,
    "share_pct": variant_counts / len(df) * 100
})

print(variant_distribution)


OVERALL VARIANT DISTRIBUTION
           users  share_pct
variant                    
control     7136  50.971429
treatment   6864  49.028571


In [44]:
print("\n" + "=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

checks = {
    "Rows": len(df),
    "Unique users": df["user_id"].nunique(),
    "Missing values": df.isnull().sum().sum(),
    "Duplicate users": df["user_id"].duplicated().sum(),
    "Invalid segments": len(invalid_segments),
    "Invalid variants": len(invalid_variants),
    "Invalid converted values": len(invalid_converted)
}

for check, value in checks.items():
    print(f"{check:25}: {value}")


print("\nAnalysis complete.")



DATA QUALITY SUMMARY
Rows                     : 14000
Unique users             : 14000
Missing values           : 0
Duplicate users          : 0
Invalid segments         : 0
Invalid variants         : 0
Invalid converted values : 0

Analysis complete.


## Question 1

In [18]:
#Q1

print("\n" + "=" * 60)
print("Q1 — OVERALL TREATMENT VS CONTROL")
print("=" * 60)

overall = (
    df.groupby("variant")["converted"]
    .agg(
        users="count",
        conversions="sum",
        conversion_rate="mean"
    )
)

overall["conversion_rate_pct"] = overall["conversion_rate"] * 100

print(overall)


control_rate = overall.loc["control", "conversion_rate_pct"]
treatment_rate = overall.loc["treatment", "conversion_rate_pct"]

naive_lift_pp = treatment_rate - control_rate

print(f"Control users   : {overall.loc['control', 'users']}")
print(f"Treatment users : {overall.loc['treatment', 'users']}")

print(f"\nControl conversion rate   : {control_rate:.4f}%")
print(f"Treatment conversion rate : {treatment_rate:.4f}%")
print(f"Naive lift                : {naive_lift_pp:.4f} percentage points")




Q1 — OVERALL TREATMENT VS CONTROL
           users  conversions  conversion_rate  conversion_rate_pct
variant                                                            
control     7136         1414         0.198150            19.815022
treatment   6864         1814         0.264277            26.427739
Control users   : 7136
Treatment users : 6864

Control conversion rate   : 19.8150%
Treatment conversion rate : 26.4277%
Naive lift                : 6.6127 percentage points


## Question 2

In [22]:
print("\n" + "=" * 60)
print("Q2 — SEGMENT LEVEL ANALYSIS")
print("=" * 60)

segment_results = (
    df.groupby(["segment", "variant"])["converted"]
    .agg(
        users="count",
        conversions="sum",
        conversion_rate="mean"
    )
    .reset_index()
)

segment_results["conversion_rate_pct"] = (
    segment_results["conversion_rate"] * 100
)

print("\nDetailed results:")
segment_results




Q2 — SEGMENT LEVEL ANALYSIS

Detailed results:


,segment,variant,users,conversions,conversion_rate,conversion_rate_pct
0,app_store,control,925,81,0.087568,8.756757
1,app_store,treatment,960,192,0.200000,20.000000
2,influencer,control,119,28,0.235294,23.529412
3,influencer,treatment,131,22,0.167939,16.793893
4,organic,control,1298,458,0.352851,35.285054
5,organic,treatment,2917,1023,0.350703,35.070278
6,paid_search,control,3353,509,0.151804,15.180435
7,paid_search,treatment,1459,210,0.143934,14.393420
8,referral,control,1441,338,0.234559,23.455933
9,referral,treatment,1397,367,0.262706,26.270580


In [21]:
# Create treatment vs control comparison table

segment_table = segment_results.pivot(
    index="segment",
    columns="variant",
    values=["users", "conversions", "conversion_rate_pct"]
)

# Flatten column names
segment_table.columns = [
    f"{metric}_{variant}"
    for metric, variant in segment_table.columns
]

segment_table = segment_table.reset_index()

# Calculate treatment - control lift
segment_table["lift_pp"] = (
    segment_table["conversion_rate_pct_treatment"]
    - segment_table["conversion_rate_pct_control"]
)

segment_table

,segment,users_control,users_treatment,conversions_control,conversions_treatment,conversion_rate_pct_control,conversion_rate_pct_treatment,lift_pp
0,app_store,925.0,960.0,81.0,192.0,8.756757,20.000000,11.243243
1,influencer,119.0,131.0,28.0,22.0,23.529412,16.793893,-6.735519
2,organic,1298.0,2917.0,458.0,1023.0,35.285054,35.070278,-0.214776
3,paid_search,3353.0,1459.0,509.0,210.0,15.180435,14.393420,-0.787015
4,referral,1441.0,1397.0,338.0,367.0,23.455933,26.270580,2.814646


In [23]:
q2_table = segment_table[
    [
        "segment",
        "users_control",
        "conversion_rate_pct_control",
        "users_treatment",
        "conversion_rate_pct_treatment",
        "lift_pp"
    ]
].copy()

q2_table.columns = [
    "Segment",
    "Control Users",
    "Control CR (%)",
    "Treatment Users",
    "Treatment CR (%)",
    "Lift (pp)"
]

q2_table

,Segment,Control Users,Control CR (%),Treatment Users,Treatment CR (%),Lift (pp)
0,app_store,925.0,8.756757,960.0,20.000000,11.243243
1,influencer,119.0,23.529412,131.0,16.793893,-6.735519
2,organic,1298.0,35.285054,2917.0,35.070278,-0.214776
3,paid_search,3353.0,15.180435,1459.0,14.393420,-0.787015
4,referral,1441.0,23.455933,1397.0,26.270580,2.814646


In [24]:

def difference_ci(control_n, control_conversions,
                  treatment_n, treatment_conversions,
                  z=1.96):
    
    control_p = control_conversions / control_n
    treatment_p = treatment_conversions / treatment_n
    
    difference = treatment_p - control_p
    
    se = np.sqrt(
        control_p * (1 - control_p) / control_n
        +
        treatment_p * (1 - treatment_p) / treatment_n
    )
    
    lower = difference - z * se
    upper = difference + z * se
    
    return difference, se, lower, upper


ci_results = []

for segment in segment_table["segment"]:
    
    row = segment_table[
        segment_table["segment"] == segment
    ].iloc[0]
    
    control_n = int(row["users_control"])
    treatment_n = int(row["users_treatment"])
    
    control_conv = int(row["conversions_control"])
    treatment_conv = int(row["conversions_treatment"])
    
    difference, se, lower, upper = difference_ci(
        control_n,
        control_conv,
        treatment_n,
        treatment_conv
    )
    
    ci_results.append({
        "segment": segment,
        "lift_pp": difference * 100,
        "standard_error_pp": se * 100,
        "CI_lower_pp": lower * 100,
        "CI_upper_pp": upper * 100
    })


ci_table = pd.DataFrame(ci_results)

ci_table

,segment,lift_pp,standard_error_pp,CI_lower_pp,CI_upper_pp
0,app_store,11.243243,1.590737,8.125398,14.361088
1,influencer,-6.735519,5.078096,-16.688588,3.217550
2,organic,-0.214776,1.593692,-3.338413,2.908860
3,paid_search,-0.787015,1.108398,-2.959475,1.385444
4,referral,2.814646,1.622477,-0.365408,5.994701


In [25]:
q2_evidence = q2_table.merge(
    ci_table,
    left_on="Segment",
    right_on="segment"
).drop(columns=["segment"])

q2_evidence

,Segment,Control Users,Control CR (%),Treatment Users,Treatment CR (%),Lift (pp),lift_pp,standard_error_pp,CI_lower_pp,CI_upper_pp
0,app_store,925.0,8.756757,960.0,20.000000,11.243243,11.243243,1.590737,8.125398,14.361088
1,influencer,119.0,23.529412,131.0,16.793893,-6.735519,-6.735519,5.078096,-16.688588,3.217550
2,organic,1298.0,35.285054,2917.0,35.070278,-0.214776,-0.214776,1.593692,-3.338413,2.908860
3,paid_search,3353.0,15.180435,1459.0,14.393420,-0.787015,-0.787015,1.108398,-2.959475,1.385444
4,referral,1441.0,23.455933,1397.0,26.270580,2.814646,2.814646,1.622477,-0.365408,5.994701


In [26]:
# Segment sample sizes

segment_sizes = (
    df.groupby(["segment", "variant"])
      .size()
      .unstack(fill_value=0)
)

segment_sizes["total"] = segment_sizes.sum(axis=1)

segment_sizes

variant,control,treatment,total
segment,,,
app_store,925,960,1885
influencer,119,131,250
organic,1298,2917,4215
paid_search,3353,1459,4812
referral,1441,1397,2838


## Question 3

In [27]:
# Q3: Population share of each segment

segment_weights = (
    df["segment"]
      .value_counts()
      .rename("users")
      .to_frame()
)

segment_weights["population_share"] = (
    segment_weights["users"] / len(df)
)

segment_weights["population_share_pct"] = (
    segment_weights["population_share"] * 100
)

segment_weights

,users,population_share,population_share_pct
segment,,,
paid_search,4812,0.343714,34.371429
organic,4215,0.301071,30.107143
referral,2838,0.202714,20.271429
app_store,1885,0.134643,13.464286
influencer,250,0.017857,1.785714


In [28]:
# Add population weights to segment results

mix_adjusted = segment_table[
    [
        "segment",
        "users_control",
        "users_treatment",
        "lift_pp"
    ]
].copy()

mix_adjusted = mix_adjusted.merge(
    segment_weights[["users", "population_share"]],
    left_on="segment",
    right_index=True
)

mix_adjusted

,segment,users_control,users_treatment,lift_pp,users,population_share
0,app_store,925.0,960.0,11.243243,1885,0.134643
1,influencer,119.0,131.0,-6.735519,250,0.017857
2,organic,1298.0,2917.0,-0.214776,4215,0.301071
3,paid_search,3353.0,1459.0,-0.787015,4812,0.343714
4,referral,1441.0,1397.0,2.814646,2838,0.202714


In [32]:
# Each segment's contribution to the overall mix-adjusted lift

mix_adjusted["weighted_lift"] = (
    mix_adjusted["lift_pp"] *
    mix_adjusted["population_share"]
)

mix_adjusted



,segment,users_control,users_treatment,lift_pp,users,population_share,weighted_lift
0,app_store,925.0,960.0,11.243243,1885,0.134643,1.513822
1,influencer,119.0,131.0,-6.735519,250,0.017857,-0.120277
2,organic,1298.0,2917.0,-0.214776,4215,0.301071,-0.064663
3,paid_search,3353.0,1459.0,-0.787015,4812,0.343714,-0.270508
4,referral,1441.0,1397.0,2.814646,2838,0.202714,0.570569


In [33]:
mix_adjusted_lift_pp = mix_adjusted["weighted_lift"].sum()

print(
    f"Mix-adjusted overall lift: "
    f"{mix_adjusted_lift_pp:.2f} percentage points"
)

Mix-adjusted overall lift: 1.63 percentage points


In [34]:
q3_table = mix_adjusted[
    [
        "segment",
        "users",
        "population_share",
        "lift_pp",
        "weighted_lift"
    ]
].copy()

q3_table["population_share_pct"] = (
    q3_table["population_share"] * 100
)

q3_table = q3_table[
    [
        "segment",
        "users",
        "population_share_pct",
        "lift_pp",
        "weighted_lift"
    ]
]

q3_table.columns = [
    "Segment",
    "Users",
    "Population Share (%)",
    "Segment Lift (pp)",
    "Weighted Contribution (pp)"
]

q3_table

,Segment,Users,Population Share (%),Segment Lift (pp),Weighted Contribution (pp)
0,app_store,1885,13.464286,11.243243,1.513822
1,influencer,250,1.785714,-6.735519,-0.120277
2,organic,4215,30.107143,-0.214776,-0.064663
3,paid_search,4812,34.371429,-0.787015,-0.270508
4,referral,2838,20.271429,2.814646,0.570569


## Question 4

In [35]:
import numpy as np

def two_proportion_ci(control_n, control_conversions,
                      treatment_n, treatment_conversions,
                      z=1.96):
    
    p_control = control_conversions / control_n
    p_treatment = treatment_conversions / treatment_n
    
    # Treatment - Control
    difference = p_treatment - p_control
    
    # Standard error
    se = np.sqrt(
        (p_control * (1 - p_control) / control_n)
        +
        (p_treatment * (1 - p_treatment) / treatment_n)
    )
    
    lower = difference - z * se
    upper = difference + z * se
    
    return difference, se, lower, upper

In [36]:
ci_results = []

for _, row in segment_table.iterrows():
    
    difference, se, lower, upper = two_proportion_ci(
        control_n=int(row["users_control"]),
        control_conversions=int(row["conversions_control"]),
        treatment_n=int(row["users_treatment"]),
        treatment_conversions=int(row["conversions_treatment"])
    )
    
    ci_results.append({
        "segment": row["segment"],
        "lift_pp": difference * 100,
        "SE_pp": se * 100,
        "CI_lower_pp": lower * 100,
        "CI_upper_pp": upper * 100
    })

ci_table = pd.DataFrame(ci_results)

ci_table

,segment,lift_pp,SE_pp,CI_lower_pp,CI_upper_pp
0,app_store,11.243243,1.590737,8.125398,14.361088
1,influencer,-6.735519,5.078096,-16.688588,3.217550
2,organic,-0.214776,1.593692,-3.338413,2.908860
3,paid_search,-0.787015,1.108398,-2.959475,1.385444
4,referral,2.814646,1.622477,-0.365408,5.994701


In [37]:
ci_display = ci_table.copy()

ci_display["95% CI"] = (
    ci_display["CI_lower_pp"].round(2).astype(str)
    + " to "
    + ci_display["CI_upper_pp"].round(2).astype(str)
    + " pp"
)

ci_display = ci_display[
    [
        "segment",
        "lift_pp",
        "95% CI"
    ]
]

ci_display

,segment,lift_pp,95% CI
0,app_store,11.243243,8.13 to 14.36 pp
1,influencer,-6.735519,-16.69 to 3.22 pp
2,organic,-0.214776,-3.34 to 2.91 pp
3,paid_search,-0.787015,-2.96 to 1.39 pp
4,referral,2.814646,-0.37 to 5.99 pp


## Question 5

In [41]:
# Q5: Treatment vs control assignment by segment

assignment = pd.crosstab(
    df["segment"],
    df["variant"]
)

assignment["total"] = assignment["control"] + assignment["treatment"]

assignment["treatment_pct"] = (
    assignment["treatment"] / assignment["total"] * 100
)

assignment["control_pct"] = (
    assignment["control"] / assignment["total"] * 100
)

assignment[
    [
        "control",
        "treatment",
        "total",
        "control_pct",
        "treatment_pct"
    ]
].round(2)

variant,control,treatment,total,control_pct,treatment_pct
segment,,,,,
app_store,925,960,1885,49.07,50.93
influencer,119,131,250,47.60,52.40
organic,1298,2917,4215,30.79,69.21
paid_search,3353,1459,4812,69.68,30.32
referral,1441,1397,2838,50.78,49.22


In [42]:
from scipy.stats import binomtest

assignment_tests = []

for segment, row in assignment.iterrows():
    
    n = int(row["total"])
    treatment = int(row["treatment"])
    
    result = binomtest(
        treatment,
        n=n,
        p=0.5,
        alternative="two-sided"
    )
    
    assignment_tests.append({
        "segment": segment,
        "total_users": n,
        "treatment_users": treatment,
        "treatment_pct": treatment / n * 100,
        "p_value": result.pvalue
    })

assignment_tests = pd.DataFrame(assignment_tests)

assignment_tests.sort_values("p_value")

,segment,total_users,treatment_users,treatment_pct,p_value
3,paid_search,4812,1459,30.320033,2.019270e-168
2,organic,4215,2917,69.205219,1.290363e-140
4,referral,2838,1397,49.224806,4.195763e-01
0,app_store,1885,960,50.928382,4.335699e-01
1,influencer,250,131,52.400000,4.866918e-01


In [43]:
# Segment composition within each variant

composition = pd.crosstab(
    df["segment"],
    df["variant"],
    normalize="columns"
) * 100

composition.round(2)

variant,control,treatment
segment,,
app_store,12.96,13.99
influencer,1.67,1.91
organic,18.19,42.50
paid_search,46.99,21.26
referral,20.19,20.35


In [2]:
from pathlib import Path
import json

answers_md = """# Onboarding Experiment Investigation

## Summary

The dataset contains 14,000 signup users across five acquisition segments and two onboarding variants. I first checked data quality, then compared conversion rates overall and within each segment, calculated a population-mix-adjusted lift, and finally checked whether treatment assignment was balanced across segments.

## Question 1 — Overall (naive) lift

| Variant | Users | Conversions | Conversion rate |
|---|---:|---:|---:|
| Control | 7,136 | 1,414 | 19.815022% |
| Treatment | 6,864 | 1,814 | 26.427739% |

Naive lift:

`26.4277389277% - 19.8150224215% = +6.6127165062 percentage points`

**Answer: +6.6127165062 percentage points**, with **7,136 control users** and **6,864 treatment users**.

## Question 2 — Segment-level comparison

| Segment | Control n | Control CR | Treatment n | Treatment CR | Lift |
|---|---:|---:|---:|---:|---:|
| app_store | 925 | 8.7568% | 960 | 20.0000% | +11.2432 pp |
| influencer | 119 | 23.5294% | 131 | 16.7939% | -6.7355 pp |
| organic | 1,298 | 35.2851% | 2,917 | 35.0703% | -0.2148 pp |
| paid_search | 3,353 | 15.1804% | 1,459 | 14.3934% | -0.7870 pp |
| referral | 1,441 | 23.4559% | 1,397 | 26.2706% | +2.8146 pp |

The segment whose estimate I would **not trust as evidence of a real effect is influencer**. Its observed treatment-control difference is -6.7355 percentage points, but the segment contains only 250 users in total (119 control and 131 treatment). Its 95% confidence interval is approximately **-16.69 to +3.22 percentage points**, so it is wide and includes zero. Thus, the large magnitude of the observed difference is not sufficient evidence of a real treatment effect.

For reference, the other 95% confidence intervals were approximately:
- app_store: **+8.13 to +14.36 pp**
- organic: **-3.34 to +2.91 pp**
- paid_search: **-2.96 to +1.39 pp**
- referral: **-0.37 to +5.99 pp**

## Question 3 — Mix-adjusted overall lift

For each segment, I calculated:

`segment lift × (segment users / 14,000)`

| Segment | Users | Population share | Segment lift | Weighted contribution |
|---|---:|---:|---:|---:|
| app_store | 1,885 | 13.4643% | +11.2432 pp | +1.5138 pp |
| influencer | 250 | 1.7857% | -6.7355 pp | -0.1203 pp |
| organic | 4,215 | 30.1071% | -0.2148 pp | -0.0647 pp |
| paid_search | 4,812 | 34.3714% | -0.7870 pp | -0.2705 pp |
| referral | 2,838 | 20.2714% | +2.8146 pp | +0.5706 pp |

Summing the weighted contributions gives:

**Mix-adjusted overall lift = +1.63 percentage points**  
(exact calculation: **+1.6289429306 pp**)

This differs from the naive +6.6127 pp because treatment and control do not have the same segment composition. In particular, treatment contains a much larger share of organic users and a much smaller share of paid-search users, and those segments have different baseline conversion rates. The mix-adjusted calculation puts both variants on the same overall population mix, separating segment-composition effects from the within-segment treatment differences.

## Question 4 — Segment with a real, meaningful positive effect

**app_store**

The evidence is:
- Control conversion rate: **8.7568%**
- Treatment conversion rate: **20.0000%**
- Lift: **+11.2432 percentage points**
- Total segment sample: **1,885 users**
- 95% confidence interval for the lift: **+8.13 to +14.36 percentage points**

The confidence interval remains above zero, and the segment has substantially more observations than the influencer segment. This makes app_store the segment with the clearest evidence of a meaningful positive treatment effect in this dataset.

The referral segment also has a positive point estimate (+2.8146 pp), but its 95% confidence interval (**-0.37 to +5.99 pp**) includes zero, so the evidence is less conclusive.

## Question 5 — Treatment assignment by segment

| Segment | Control | Treatment | Control % | Treatment % |
|---|---:|---:|---:|---:|
| app_store | 925 | 960 | 49.07% | 50.93% |
| influencer | 119 | 131 | 47.60% | 52.40% |
| organic | 1,298 | 2,917 | **30.79%** | **69.21%** |
| paid_search | 3,353 | 1,459 | **69.68%** | **30.32%** |
| referral | 1,441 | 1,397 | 50.78% | 49.22% |

The overall experiment is close to 50/50, but assignment is strongly imbalanced within two segments: organic is 69.21% treatment, while paid_search is only 30.32% treatment. Under a 50/50-per-segment assignment assumption, these deviations are far too large to look like ordinary random variation; the other three segments are close to 50/50. This assignment imbalance is important because organic and paid-search users have very different baseline conversion rates and therefore can materially affect the naive overall comparison.

A binomial test against a 50% treatment probability gave extremely small two-sided p-values for organic and paid_search, reinforcing that the observed splits would be highly unusual under that specific 50/50 assumption. This does not by itself prove that the experiment was not randomized, because the actual assignment mechanism is not provided; it does establish that the treatment/control mix is highly segment-dependent.

## Investigation process

- Loaded the CSV with pandas and verified the dataset shape: 14,000 rows and 4 columns.
- Checked for missing values; none were present in the data used for the analysis.
- Checked for duplicate rows and duplicate user IDs; no duplicates were found.
- Validated the allowed segment, variant, and conversion values.
- Calculated the overall treatment/control conversion rates and the naive lift.
- Broke conversion rates down by acquisition segment and calculated treatment-control lifts.
- Checked confidence intervals for the segment-level differences to distinguish large point estimates from well-supported effects.
- Initially, the overall +6.61 pp result looked like the main conclusion, but this was a dead end because it ignored the different segment composition of treatment and control.
- Calculated the mix-adjusted lift using each segment's share of the full 14,000-user population.
- Checked treatment/control assignment shares within each segment and tested the observed deviations from a 50/50 allocation assumption.
"""

answers_json = {
    "q1_naive_lift_pp": 6.612716506214261,
    "q1_n_control": 7136,
    "q1_n_treatment": 6864,
    "q2_untrustworthy_segment": "influencer",
    "q3_mix_adjusted_lift_pp": 1.6289429305586989,
    "q4_real_effect_segment": "app_store"
}

Path("../ANSWERS.md").write_text(answers_md, encoding="utf-8")
Path("../answers.json").write_text(
    json.dumps(answers_json, indent=2) + "\n",
    encoding="utf-8"
)

print("Created:")
print("/mnt/data/ANSWERS.md")
print("/mnt/data/answers.json")
print("\nanswers.json:")
print(json.dumps(answers_json, indent=2))


Created:
/mnt/data/ANSWERS.md
/mnt/data/answers.json

answers.json:
{
  "q1_naive_lift_pp": 6.612716506214261,
  "q1_n_control": 7136,
  "q1_n_treatment": 6864,
  "q2_untrustworthy_segment": "influencer",
  "q3_mix_adjusted_lift_pp": 1.6289429305586989,
  "q4_real_effect_segment": "app_store"
}
